[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/48_int8_quantization.ipynb)

# 🟡 Medium: INT8 Quantization (symmetric and asymmetric)

*Inference & Decoding*
Implement per-tensor **int8 quantization**, both symmetric and asymmetric.

**Symmetric** — one parameter, zero maps exactly to zero:

$$s = \frac{\max|x|}{127}, \quad z = 0, \quad q = \text{clip}\!\left(\text{round}\!\left(\tfrac{x}{s}\right), -128, 127\right)$$

**Asymmetric** — two parameters, uses the full range on skewed data:

$$s = \frac{\max(x) - \min(x)}{255}, \quad
z = \text{round}\!\left(\tfrac{-\min(x)}{s}\right) - 128, \quad
q = \text{clip}\!\left(\text{round}\!\left(\tfrac{x}{s}\right) + z, -128, 127\right)$$

Dequantization is the same in both cases: $\hat{x} = (q - z)\,s$.

### Signature
```python
def quantize_int8(x, symmetric=True):
    ...  # -> (q, scale, zero_point, x_dequant)
```
`q` must be `int8`. `scale` is a scalar float, `zero_point` a scalar int.

### Rules
- Per-**tensor** (one scale for the whole array), not per-channel
- Clip into `[-128, 127]` before casting
- Handle a constant tensor (zero range) without producing `NaN`/`Inf`
- `round` must be round-half-to-even (`jnp.round`), matching hardware

### Symmetric vs asymmetric
Symmetric is cheaper: with $z=0$, a quantized matmul is just an integer matmul
times a scale. Asymmetric needs cross-terms and extra bookkeeping. But on skewed
data — post-ReLU activations are all $\ge 0$ — symmetric throws away half the
range, since nothing ever maps below 0. Rule of thumb: **symmetric for weights**
(roughly zero-centred), **asymmetric for activations**.

### The thing that actually breaks LLM quantization
Per-tensor quantization is hostage to a single number: $\max|x|$. Transformer
activations famously contain a handful of **outlier channels** with magnitudes
20-100x everything else, and they consistently appear in the same feature
dimensions. One outlier inflates the scale, and every ordinary value collapses
into a couple of quantization levels — accuracy falls off a cliff, and it gets
*worse* as models get bigger.

That single observation is what the whole modern literature is built around:
LLM.int8() splits the outlier channels out into fp16, SmoothQuant migrates the
scale difficulty from activations into weights, AWQ keeps the salient channels
at higher precision. The tests below reproduce the failure directly so you can
see the magnitude of it.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def quantize_int8(x, symmetric=True):
    """Per-tensor int8 quantization.

    Args:
        x:         float array
        symmetric: True for symmetric (zero_point = 0), False for asymmetric

    Returns:
        (q, scale, zero_point, x_dequant)
          q          int8 array, same shape as x
          scale      python/JAX float scalar
          zero_point python/JAX int scalar
          x_dequant  float array, the round-trip reconstruction
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

x = jax.random.normal(jax.random.key(0), (1000,))
_, s, z, xd = quantize_int8(x, symmetric=True)
print(f"clean:    scale={float(s):.5f}  max err={float(jnp.abs(x - xd).max()):.5f}")

# Now plant a single outlier, as real transformer activations have.
x_out = x.at[0].set(50.0)
_, s2, _, xd2 = quantize_int8(x_out, symmetric=True)
err = float(jnp.abs(x_out[1:] - xd2[1:]).max())
print(f"1 outlier: scale={float(s2):.5f}  max err on the OTHER 999 values={err:.5f}")
print(f"-> one value inflated the error on everything else by {err / float(jnp.abs(x - xd).max()):.0f}x")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("int8_quantization")

# hint("int8_quantization")      # stuck? nudge without the answer
# solution("int8_quantization")  # spoiler: the reference implementation